# Serve clef-flash-tpu-int8 on a Colab TPU

Serves [heisvibhor-1/clef-flash-tpu-int8](https://huggingface.co/heisvibhor-1/clef-flash-tpu-int8) with the same HTTP API as Cloudflare's clef-flash server:
- `POST /v1/systemone`: a [Jev / SystemOne](https://huggingface.co/Cloudflare/clef-flash#jev--systemone-api) request body in, the same response body out (`model`, `answers`, `usage`).
- `GET /health`: `{"status": "ok"}`.

Requests run one at a time, behind a `cloudflared` quick tunnel that gives the server a public URL.

**Runtime:** Runtime > Change runtime type > **v5e-1 TPU**. Run the cells top to bottom.

## Public tunnel

Starts first, so you can point your client at the URL while the model loads. Until the server cell has run, the URL answers HTTP 502.

The quick tunnel has no authentication: the `trycloudflare.com` URL is unguessable but public, and it changes every run.

In [ ]:
import re
import subprocess
import time

PORT = 8090
LOCAL_URL = f"http://127.0.0.1:{PORT}"
TUNNEL_LOG = "/content/cloudflared.log"
TUNNEL_URL_PATTERN = re.compile(r"https://[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com")

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", LOCAL_URL],
    stdout=open(TUNNEL_LOG, "w"),
    stderr=subprocess.STDOUT,
)


def wait_for_tunnel(timeout_s=90):
    """The public URL, once cloudflared has connected to Cloudflare's edge."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        log = open(TUNNEL_LOG).read()
        match = TUNNEL_URL_PATTERN.search(log)
        if match and "Registered tunnel connection" in log:
            return match.group(0)
        if tunnel.poll() is not None:
            raise RuntimeError(f"cloudflared exited with {tunnel.returncode}; see {TUNNEL_LOG}")
        time.sleep(1)
    raise RuntimeError(f"no tunnel after {timeout_s}s; see {TUNNEL_LOG}")


ENDPOINT_URL = f"{wait_for_tunnel()}/v1/systemone"
print(f"endpoint: {ENDPOINT_URL}")

In [ ]:
import importlib.util
import sys

MODEL_REPO = "heisvibhor-1/clef-flash-tpu-int8"

# joint_schema_model.py imports torch; only its prompt builder and answer format run here.
if importlib.util.find_spec("torch") is None:
    %pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q -U "transformers>=5.10" safetensors

from huggingface_hub import HfApi, snapshot_download

REVISION = HfApi().model_info(MODEL_REPO).sha
sys.path.insert(0, snapshot_download(MODEL_REPO, revision=REVISION, ignore_patterns=["compiled/*"]))
import clef_tpu

# compiled/ has one folder per TPU type and jax/libtpu build; only this runtime's folder can load.
RUNTIME_ID = clef_tpu.runtime_id()
print(f"runtime {RUNTIME_ID}:\n{clef_tpu.runtime_description()}")
snapshot_download(MODEL_REPO, revision=REVISION, allow_patterns=[f"compiled/{RUNTIME_ID}/*"])

started = time.perf_counter()
model = clef_tpu.load_model()
print(f"loaded in {time.perf_counter() - started:.0f}s, with {len(model.compiled_backbones)} compiled backbones")

## Compiled backbones

Prompts are padded to a multiple of 1,024 tokens, and each length needs its own compiled backbone (about 22 s each). The repo ships them under `compiled/<runtime id>/`, so on a known runtime this cell only checks that every bucket up to `WARM_UP_TOKENS` was loaded, and finishes in about a second.

On a runtime the repo has no folder for (a new Colab image or jax version), it compiles the missing buckets (about 3 minutes) and saves them to `COMPILED_DIR`. `UPLOAD_COMPILED = True` uploads them to `MODEL_REPO` for the next session, which needs write access to it. Each new question count still compiles the decision head once, which takes a few seconds.

`SERVE_INT8_ACTIVATIONS = True` serves W8A8, which also rounds activations to int8: on ARC-Challenge it was 26% faster and changed 9 of 1,172 answers (see the model card). Set it to `False` for int8 weights with BF16 activations.

In [ ]:
SERVE_INT8_ACTIVATIONS = True
# A longer prompt compiles its bucket on its first request.
WARM_UP_TOKENS = 8192
UPLOAD_COMPILED = False
COMPILED_DIR = f"/content/compiled/{RUNTIME_ID}"

started = time.perf_counter()
new_buckets = clef_tpu.compile_backbones(model, WARM_UP_TOKENS, SERVE_INT8_ACTIVATIONS, save_dir=COMPILED_DIR)
print(f"compiled {len(new_buckets)} missing buckets in {time.perf_counter() - started:.0f}s: {new_buckets}")

if new_buckets and UPLOAD_COMPILED:
    HfApi().upload_folder(
        repo_id=MODEL_REPO,
        folder_path=COMPILED_DIR,
        path_in_repo=f"compiled/{RUNTIME_ID}",
        commit_message=f"Add compiled backbones for runtime {RUNTIME_ID}",
    )
    print(f"uploaded to {MODEL_REPO}/compiled/{RUNTIME_ID}")

In [ ]:
import json
import threading
import traceback
from http.server import BaseHTTPRequestHandler, HTTPServer

import requests

SERVER_LOG = "/content/systemone-server.log"


def log_line(text):
    with open(SERVER_LOG, "a") as log:
        log.write(f"{time.strftime('%H:%M:%S')} {text}\n")


class SystemOneHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/health":
            self.send_json(200, {"status": "ok"})
        else:
            self.send_json(404, {"error": f"no route {self.path}"})

    def do_POST(self):
        if self.path != "/v1/systemone":
            return self.send_json(404, {"error": f"no route {self.path}"})
        try:
            request = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            body = clef_tpu.systemone(model, request, SERVE_INT8_ACTIVATIONS)
        except (KeyError, TypeError, ValueError) as error:
            return self.send_json(400, {"error": f"{type(error).__name__}: {error}"})
        except Exception as error:
            log_line(traceback.format_exc())
            return self.send_json(500, {"error": f"{type(error).__name__}: {error}"})
        self.send_json(200, body)

    def send_json(self, status, body):
        payload = json.dumps(body, ensure_ascii=False).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def log_message(self, format, *args):
        log_line(format % args)


server = HTTPServer(("127.0.0.1", PORT), SystemOneHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

sample_request = {
    "model": "clef-flash",
    "state": "Our checkout started returning errors and orders are blocked.",
    "questions": {
        "department": {
            "type": "choice",
            "instructions": "Which team should handle the message?",
            "criteria": {"billing": "Payments or invoices", "technical": "Bugs or outages"},
        },
        "outage": {"type": "noul", "instructions": "Is a service down?"},
    },
}

assert requests.get(f"{LOCAL_URL}/health", timeout=5).ok
local_response = requests.post(f"{LOCAL_URL}/v1/systemone", json=sample_request, timeout=120)
local_response.raise_for_status()
print(json.dumps(local_response.json(), indent=2, ensure_ascii=False))

requests.post(ENDPOINT_URL, json=sample_request, timeout=120).raise_for_status()
print(f"serving at {ENDPOINT_URL}")

## Notes

- **Keep this tab open:** free Colab disconnects idle sessions, and the endpoint goes with it. After a reconnect, run all cells again; the URL changes.
- **Request log:** every request and any server-side traceback goes to `/content/systemone-server.log` (`!tail /content/systemone-server.log`).
- **Errors:** a malformed request gets HTTP 400 and a model failure HTTP 500, both as `{"error": ...}`.
- **One request at a time:** concurrent requests wait their turn, so a client's timeout has to cover the queue too.
- **Long prompts:** states over 16,384 tokens are cut, as in Cloudflare's code.
- **Stop** by uncommenting and running the next cell. It's commented out so Run all leaves the server up.

In [ ]:
# tunnel.terminate()
# server.shutdown()